In [26]:
import pandas as pd
import numpy as np
import os
from pathlib import Path

BASE_V2010 = Path('../hcris-data/HCRIS_v2010')
BASE_V1996 = Path('../hcris-data/HCRIS_v1996')
OUT_DIR    = Path('data/output')
OUT_DIR.mkdir(parents=True, exist_ok=True)

pd.set_option('display.float_format', '{:,.2f}'.format)
print('Setup complete.')
print(f'v2010 path: {BASE_V2010.resolve()}')
print(f'v1996 path: {BASE_V1996.resolve()}')
print(f'Output dir: {OUT_DIR.resolve()}')

Setup complete.
v2010 path: /scion/5261/econ470001/hcris-data/HCRIS_v2010
v1996 path: /scion/5261/econ470001/hcris-data/HCRIS_v1996
Output dir: /home/rpat638/econ470/a0/work/hwk4/data/output


In [27]:
VAR_MAP_V2010 = {
    'tot_charges':         ('G300000',   100,  '00100'),
    'tot_discounts':       ('G300000',   200,  '00100'),
    'ip_charges':          ('C000001', 20200,  '00100'),
    'icu_charges':         ('C000001', 20100,  '00100'),
    'ancillary_charges':   ('C000191', 20000,  '00100'),
    'tot_mcare_payment':   ('E00A18A',  7200,  '00100'),
    'mcare_discharges':    ('S300001',  1400,  '00800'),
    'tot_discharges':      ('S300001',  1400,  '00300'),
    'beds':                ('S300001',   100,  '00100'),
    'medicaid_discharges': ('S300001',  1400,  '00700'),
    'hrrp_payment':        ('E00A18A',  7400,  '00100'),
    'vbp_payment':         ('E00A18B',  4300,  '00100'),
}

VAR_MAP_V1996 = {
    'tot_charges':         ('G300000',  100,  '0100'),
    'tot_discounts':       ('G300000',  200,  '0100'),
    'tot_mcare_payment':   ('E00A18A', 2800,  '0100'),
    'mcare_discharges':    ('S300001', 1200,  '0400'),
    'tot_discharges':      ('S300001', 1200,  '0200'),
    'beds':                ('S300001',  100,  '0100'),
    'medicaid_discharges': ('S300001', 1200,  '0500'),
}

print('Variable maps defined.')
print(f'v2010 variables: {list(VAR_MAP_V2010.keys())}')
print(f'v1996 variables: {list(VAR_MAP_V1996.keys())}')

Variable maps defined.
v2010 variables: ['tot_charges', 'tot_discounts', 'ip_charges', 'icu_charges', 'ancillary_charges', 'tot_mcare_payment', 'mcare_discharges', 'tot_discharges', 'beds', 'medicaid_discharges', 'hrrp_payment', 'vbp_payment']
v1996 variables: ['tot_charges', 'tot_discounts', 'tot_mcare_payment', 'mcare_discharges', 'tot_discharges', 'beds', 'medicaid_discharges']


In [28]:
def load_rpt(path):
    rpt = pd.read_csv(path, header=None, low_memory=False)
    rpt.columns = [
        'rpt_rec_num', 'wksht_cd', 'prvdr_num', 'npi',
        'rpt_stus_cd', 'fy_bgn_dt', 'fy_end_dt', 'proc_dt',
        'initl_rpt_sw', 'last_rpt_sw', 'trnsmtl_cd', 'fi_num',
        'adr_vndr_cd', 'fi_creat_dt', 'util_cd', 'npr_dt',
        'spec_ind', 'fi_rcpt_dt'
    ]
    rpt['rpt_rec_num'] = pd.to_numeric(rpt['rpt_rec_num'], errors='coerce')
    rpt['prvdr_num']   = rpt['prvdr_num'].astype(str).str.strip().str.zfill(6)
    rpt['fy_end_dt']   = pd.to_datetime(rpt['fy_end_dt'], errors='coerce')
    rpt['year']        = rpt['fy_end_dt'].dt.year
    return rpt[['rpt_rec_num', 'prvdr_num', 'year', 'util_cd',
                'fy_bgn_dt', 'fy_end_dt']].dropna(subset=['rpt_rec_num'])

test_rpt = load_rpt(BASE_V2010 / 'HospitalFY2012' / 'hosp10_2012_RPT.CSV')
print(f'2012 RPT rows: {len(test_rpt):,}')
print(test_rpt.head(3).to_string())

2012 RPT rows: 6,202
   rpt_rec_num prvdr_num  year util_cd   fy_bgn_dt  fy_end_dt
0         7080    110050  2012       F  10/01/2011 2012-01-31
1         7716    330409  2011       L  10/17/2011 2011-12-31
2         7952    150178  2012       N  01/01/2012 2012-03-23


In [29]:
def extract_vars(nmrc_path, rpt_df, var_map):
    targets = set()
    for ws, line, col in var_map.values():
        targets.add((ws, line, col))

    nmrc = pd.read_csv(
        nmrc_path, header=None, low_memory=False,
        names=['rpt_rec_num', 'worksheet', 'line', 'col', 'value']
    )
    nmrc['rpt_rec_num'] = pd.to_numeric(nmrc['rpt_rec_num'], errors='coerce')
    nmrc['line']        = pd.to_numeric(nmrc['line'],        errors='coerce')
    nmrc['value']       = pd.to_numeric(nmrc['value'],       errors='coerce')
    nmrc['col']         = nmrc['col'].astype(str).str.strip()

    nmrc['key'] = list(zip(nmrc['worksheet'], nmrc['line'], nmrc['col']))
    nmrc_sub    = nmrc[nmrc['key'].isin(targets)].copy()

    reverse_map      = {v: k for k, v in var_map.items()}
    nmrc_sub['var_name'] = nmrc_sub['key'].map(reverse_map)
    nmrc_sub = nmrc_sub.dropna(subset=['var_name'])

    valid_rpts = set(rpt_df['rpt_rec_num'].dropna())
    nmrc_sub   = nmrc_sub[nmrc_sub['rpt_rec_num'].isin(valid_rpts)]

    wide = (
        nmrc_sub
        .groupby(['rpt_rec_num', 'var_name'])['value']
        .first()
        .unstack('var_name')
        .reset_index()
    )

    result = rpt_df.merge(wide, on='rpt_rec_num', how='left')
    return result

print('extract_vars() defined.')

extract_vars() defined.


In [30]:
def get_v2010_paths(year):
    folder = BASE_V2010 / f'HospitalFY{year}'
    nmrc_candidates = [
        folder / f'hosp10_{year}_NMRC.CSV',
        folder / f'HOSP10_{year}_nmrc.csv',
    ]
    rpt_candidates = [
        folder / f'hosp10_{year}_RPT.CSV',
        folder / f'HOSP10_{year}_rpt.csv',
    ]
    nmrc_path = next((p for p in nmrc_candidates if p.exists()), None)
    rpt_path  = next((p for p in rpt_candidates  if p.exists()), None)
    return nmrc_path, rpt_path

V2010_YEARS    = list(range(2010, 2020))
all_years_v2010 = []

for yr in V2010_YEARS:
    nmrc_path, rpt_path = get_v2010_paths(yr)
    if nmrc_path is None or rpt_path is None:
        print(f'SKIPPING {yr} — files not found')
        continue
    print(f'Processing v2010 year {yr}...', end=' ')
    rpt_df = load_rpt(rpt_path)
    df_yr  = extract_vars(nmrc_path, rpt_df, VAR_MAP_V2010)
    all_years_v2010.append(df_yr)
    print(f'{len(df_yr):,} reports')

hcris_v2010 = pd.concat(all_years_v2010, ignore_index=True)
print(f'\nv2010 combined: {len(hcris_v2010):,} rows')
print(f'Years present: {sorted(hcris_v2010["year"].dropna().astype(int).unique())}')

Processing v2010 year 2010... 2,322 reports
Processing v2010 year 2011... 6,143 reports
Processing v2010 year 2012... 6,202 reports
Processing v2010 year 2013... 6,144 reports
Processing v2010 year 2014... 6,190 reports
Processing v2010 year 2015... 6,199 reports
Processing v2010 year 2016... 6,204 reports
Processing v2010 year 2017... 6,121 reports
Processing v2010 year 2018... 6,159 reports
Processing v2010 year 2019... 6,118 reports

v2010 combined: 57,802 rows
Years present: [np.int64(2011), np.int64(2012), np.int64(2013), np.int64(2014), np.int64(2015), np.int64(2016), np.int64(2017), np.int64(2018), np.int64(2019), np.int64(2020)]


In [31]:
def get_v1996_paths(year):
    folder    = BASE_V1996 / f'HospitalFY{year}'
    nmrc_path = folder / f'hosp_{year}_NMRC.CSV'
    rpt_path  = folder / f'hosp_{year}_RPT.CSV'
    return (
        nmrc_path if nmrc_path.exists() else None,
        rpt_path  if rpt_path.exists()  else None
    )

print('Processing v1996 year 2009...', end=' ')
nmrc_09, rpt_09 = get_v1996_paths(2009)

if nmrc_09 and rpt_09:
    rpt_df_09   = load_rpt(rpt_09)
    hcris_v1996 = extract_vars(nmrc_09, rpt_df_09, VAR_MAP_V1996)
    for col in ['ip_charges', 'icu_charges', 'ancillary_charges',
                'hrrp_payment', 'vbp_payment']:
        hcris_v1996[col] = np.nan
    print(f'{len(hcris_v1996):,} reports')
else:
    print('FILES NOT FOUND')
    hcris_v1996 = pd.DataFrame()

print(hcris_v1996[['prvdr_num', 'year', 'mcare_discharges',
                    'tot_charges', 'beds']].head(3).to_string())

Processing v1996 year 2009... 6,196 reports
  prvdr_num  year  mcare_discharges   tot_charges   beds
0    220025  2009            764.00 23,835,841.00  16.00
1    050714  2009            132.00 36,000,406.00  30.00
2    010157  2009            760.00 24,519,761.00 112.00


In [32]:
hcris_v2010 = hcris_v2010.rename(columns={'prvdr_num': 'provider_number'})
hcris_v1996 = hcris_v1996.rename(columns={'prvdr_num': 'provider_number'})

hcris_all = pd.concat([hcris_v1996, hcris_v2010], ignore_index=True)

hcris_all = hcris_all.dropna(subset=['year', 'provider_number'])
hcris_all['year'] = hcris_all['year'].astype(int)
hcris_all = hcris_all[hcris_all['year'].between(2009, 2019)]

print(f'Combined rows:     {len(hcris_all):,}')
print(f'Years:             {sorted(hcris_all["year"].unique())}')
print(f'Unique providers:  {hcris_all["provider_number"].nunique():,}')

Combined rows:     61,401
Years:             [np.int64(2009), np.int64(2010), np.int64(2011), np.int64(2012), np.int64(2013), np.int64(2014), np.int64(2015), np.int64(2016), np.int64(2017), np.int64(2018), np.int64(2019)]
Unique providers:  6,850


In [33]:
report_counts = (
    hcris_all
    .groupby(['provider_number', 'year'])
    .size()
    .reset_index(name='n_reports')
)

all_years = pd.DataFrame({'year': list(range(2009, 2020))})

dup_by_year = (
    report_counts[report_counts['n_reports'] > 1]
    .groupby('year')['provider_number']
    .nunique()
    .reset_index(name='n_hospitals_dup')
)

dup_by_year = (
    all_years
    .merge(dup_by_year, on='year', how='left')
    .fillna(0)
    .astype({'n_hospitals_dup': int})
)

print('=== HOSPITALS WITH DUPLICATE REPORTS BY YEAR ===')
print(dup_by_year.to_string(index=False))

dup_by_year.to_csv(OUT_DIR / 'dup_hospitals_by_year.csv', index=False)
print('\nSaved: dup_hospitals_by_year.csv')

=== HOSPITALS WITH DUPLICATE REPORTS BY YEAR ===
 year  n_hospitals_dup
 2009               37
 2010                0
 2011               45
 2012               90
 2013               91
 2014              102
 2015               96
 2016               95
 2017               91
 2018              116
 2019              101

Saved: dup_hospitals_by_year.csv


In [34]:
hcris_all['util_priority'] = hcris_all['util_cd'].map(
    {'F': 3, 'L': 2, 'N': 1}
).fillna(0).astype(int)

hcris = (
    hcris_all
    .sort_values(
        ['provider_number', 'year', 'util_priority', 'rpt_rec_num'],
        ascending=[True, True, False, False]
    )
    .drop_duplicates(subset=['provider_number', 'year'], keep='first')
    .drop(columns=['util_priority'])
    .reset_index(drop=True)
)

print(f'Rows after dedup:      {len(hcris):,}')
print(f'Unique hospitals : {hcris["provider_number"].nunique():,}')

Rows after dedup:      60,518
Unique hospitals : 6,850


In [35]:
unique_hospitals = hcris['provider_number'].nunique()
print(f'Unique hospital IDs: {unique_hospitals:,}')

pd.DataFrame({'unique_hospitals': [unique_hospitals]}).to_csv(
    OUT_DIR / 'unique_hospitals.csv', index=False
)
print('Saved: unique_hospitals.csv')

Unique hospital IDs: 6,850
Saved: unique_hospitals.csv


In [36]:
abs_vars = [
    'tot_charges', 'tot_discounts',
    'ip_charges', 'icu_charges', 'ancillary_charges',
    'tot_mcare_payment', 'tot_discharges',
    'mcare_discharges', 'medicaid_discharges', 'beds'
]

for col in abs_vars:
    if col in hcris.columns:
        hcris[col] = pd.to_numeric(hcris[col], errors='coerce').abs()

if 'hrrp_payment' in hcris.columns:
    hcris['hrrp_payment'] = pd.to_numeric(
        hcris['hrrp_payment'], errors='coerce'
    ).clip(lower=0)

if 'vbp_payment' in hcris.columns:
    hcris['vbp_payment'] = pd.to_numeric(
        hcris['vbp_payment'], errors='coerce'
    )

print('Negative value handling complete.')
print(hcris[['tot_charges', 'mcare_discharges', 'hrrp_payment']].describe().round(0))

Negative value handling complete.
            tot_charges  mcare_discharges  hrrp_payment
count         58,150.00         59,711.00     29,257.00
mean     511,808,772.00         28,023.00    390,845.00
std    1,010,723,269.00         46,221.00  1,236,343.00
min                3.00              1.00          0.00
25%       36,859,751.00          3,383.00          0.00
50%      122,589,474.00         10,883.00     23,286.00
75%      564,574,472.00         33,965.00    275,405.00
max   22,000,932,119.00      3,500,849.00 46,825,984.00


In [37]:
hcris = hcris.assign(
    discount_factor = lambda d:
        1 - d['tot_discounts'] / d['tot_charges'],

    price_num = lambda d:
        (d['ip_charges'].fillna(0)
         + d['icu_charges'].fillna(0)
         + d['ancillary_charges'].fillna(0))
        * (1 - d['tot_discounts'] / d['tot_charges'])
        - d['tot_mcare_payment'],

    price_denom = lambda d:
        d['tot_discharges'] - d['mcare_discharges'],

    price = lambda d:
        (
            (d['ip_charges'].fillna(0)
             + d['icu_charges'].fillna(0)
             + d['ancillary_charges'].fillna(0))
            * (1 - d['tot_discounts'] / d['tot_charges'])
            - d['tot_mcare_payment']
        ) / (d['tot_discharges'] - d['mcare_discharges'])
)

hcris['price'] = hcris['price'].replace([np.inf, -np.inf], np.nan)

print('Price computed.')
print(hcris['price'].describe().round(2))
print(f'NaN prices: {hcris["price"].isna().sum():,}')

Price computed.
count       32,845.00
mean         3,898.08
std         76,377.13
min     -1,033,183.16
25%            486.03
50%          1,595.22
75%          3,595.12
max     11,828,969.85
Name: price, dtype: float64
NaN prices: 27,673


In [38]:
hcris['hrrp_penalty'] = hcris['hrrp_payment'].fillna(0)

hcris['vbp_penalty'] = np.where(
    hcris['vbp_payment'].fillna(0) < 0,
    -hcris['vbp_payment'].fillna(0),
    0.0
)

hcris['net_penalty'] = hcris['hrrp_penalty'] + hcris['vbp_penalty']

hcris['any_penalty'] = (
    (hcris['hrrp_penalty'] > 0) | (hcris['vbp_penalty'] > 0)
).astype(int)

print('Penalty variables created.')
yr = hcris[hcris['year'] == 2012]
print(f'\n2012 hospitals:    {len(yr):,}')
print(f'HRRP penalized:    {(yr["hrrp_penalty"] > 0).sum():,}')
print(f'VBP penalized:     {(yr["vbp_penalty"] > 0).sum():,}')
print(f'Any penalized:     {yr["any_penalty"].sum():,}')

Penalty variables created.

2012 hospitals:    6,140
HRRP penalized:    1,888
VBP penalized:     2,138
Any penalized:     3,395


In [39]:
hcris.to_csv(OUT_DIR / 'hcris_clean.csv', index=False)

print(f'Saved hcris_clean.csv')
print(f'  Rows:         {len(hcris):,}')
print(f'  Unique hosps: {hcris["provider_number"].nunique():,}')
print(f'  Years:        {sorted(hcris["year"].unique())}')
print(f'  Columns:      {list(hcris.columns)}')

Saved hcris_clean.csv
  Rows:         60,518
  Unique hosps: 6,850
  Years:        [np.int64(2009), np.int64(2010), np.int64(2011), np.int64(2012), np.int64(2013), np.int64(2014), np.int64(2015), np.int64(2016), np.int64(2017), np.int64(2018), np.int64(2019)]
  Columns:      ['rpt_rec_num', 'provider_number', 'year', 'util_cd', 'fy_bgn_dt', 'fy_end_dt', 'beds', 'mcare_discharges', 'medicaid_discharges', 'tot_charges', 'tot_discharges', 'tot_discounts', 'tot_mcare_payment', 'ip_charges', 'icu_charges', 'ancillary_charges', 'hrrp_payment', 'vbp_payment', 'discount_factor', 'price_num', 'price_denom', 'price', 'hrrp_penalty', 'vbp_penalty', 'net_penalty', 'any_penalty']


In [40]:
share_pen = (
    hcris[hcris['year'].between(2012, 2019)]
    .groupby('year', as_index=False)
    .agg(
        n_hospitals = ('provider_number', 'count'),
        share_hrrp  = ('hrrp_penalty',    lambda x: (x > 0).mean()),
        share_vbp   = ('vbp_penalty',     lambda x: (x > 0).mean()),
        share_any   = ('any_penalty',     'mean'),
    )
)

print('=== SHARE PENALIZED BY YEAR  ===')
print(share_pen.to_string(index=False))

share_pen.to_csv(OUT_DIR / 'share_penalized_by_year.csv', index=False)
print('\nSaved: share_penalized_by_year.csv')

=== SHARE PENALIZED BY YEAR  ===
 year  n_hospitals  share_hrrp  share_vbp  share_any
 2012         6140        0.31       0.35       0.55
 2013         6066        0.34       0.30       0.55
 2014         6064        0.36       0.28       0.56
 2015         6051        0.36       0.28       0.55
 2016         6091        0.33       0.29       0.53
 2017         6083        0.30       0.27       0.49
 2018         6042        0.29       0.29       0.51
 2019         6031        0.32       0.28       0.51

Saved: share_penalized_by_year.csv


In [41]:
prices = (
    hcris[hcris['year'].isin([2011, 2014])]
    [['provider_number', 'year', 'price']]
    .pivot(index='provider_number', columns='year', values='price')
    .rename(columns={2011: 'price_2011', 2014: 'price_2014'})
    .reset_index()
)

pen_2012 = (
    hcris[hcris['year'] == 2012]
    [['provider_number', 'net_penalty', 'hrrp_penalty',
      'vbp_penalty', 'any_penalty']]
    .copy()
)
pen_2012['net_penalty_1k']  = pen_2012['net_penalty']  / 1_000
pen_2012['hrrp_penalty_1k'] = pen_2012['hrrp_penalty'] / 1_000

pre = hcris[hcris['year'].between(2009, 2011)]

pre_avgs = (
    pre
    .groupby('provider_number', as_index=False)
    .agg(
        avg_mcare_dis = ('mcare_discharges',   'mean'),
        avg_beds      = ('beds',                'mean'),
        avg_mcaid_dis = ('medicaid_discharges', 'mean'),
    )
)

pre_avgs['avg_mcare_100'] = pre_avgs['avg_mcare_dis'] / 100

print(f'Price pairs (2011 & 2014): {len(prices):,}')
print(f'2012 penalties:            {len(pen_2012):,}')
print(f'Pre-period averages:       {len(pre_avgs):,}')

Price pairs (2011 & 2014): 6,307
2012 penalties:            6,140
Pre-period averages:       6,261


In [42]:
iv_data = (
    prices
    .merge(pen_2012, on='provider_number', how='left')
    .merge(pre_avgs,  on='provider_number', how='left')
)

iv_data['price_change'] = iv_data['price_2014'] - iv_data['price_2011']

for col in ['net_penalty', 'net_penalty_1k', 'hrrp_penalty',
            'hrrp_penalty_1k', 'vbp_penalty', 'any_penalty']:
    if col in iv_data.columns:
        iv_data[col] = iv_data[col].fillna(0)

iv_clean = iv_data.dropna(
    subset=['price_2011', 'price_2014', 'avg_mcare_100']
).copy()

iv_clean = iv_clean[
    (iv_clean['price_2011'] > 0) &
    (iv_clean['price_2014'] > 0)
].copy()

for col in ['price_2011', 'price_2014']:
    p05 = iv_clean[col].quantile(0.05)
    p95 = iv_clean[col].quantile(0.95)
    iv_clean[col] = iv_clean[col].clip(lower=p05, upper=p95)

iv_clean['price_change'] = iv_clean['price_2014'] - iv_clean['price_2011']

print(f'IV dataset rows: {len(iv_clean):,}')
print(iv_clean[['price_change', 'net_penalty_1k',
                'avg_mcare_100', 'avg_beds',
                'avg_mcaid_dis']].describe().round(2))

IV dataset rows: 2,936
       price_change  net_penalty_1k  avg_mcare_100   avg_beds  avg_mcaid_dis
count      2,936.00        2,936.00       2,936.00   2,936.00       2,868.00
mean         284.25          438.79         307.14     138.05       6,699.35
std        1,767.84        1,447.83         348.49   2,111.51      11,030.90
min       -8,571.02            0.00           0.33      17.00           1.00
25%         -411.25            0.00          70.00      44.50         916.12
50%           32.70           44.47         192.25      75.00       2,778.50
75%          818.62          293.00         423.66     126.00       7,586.62
max       10,028.65       23,614.86       3,951.18 114,128.00     148,166.00


In [43]:
iv_clean.to_csv(OUT_DIR / 'hcris_iv.csv', index=False)
print(f'Saved hcris_iv.csv')
print(f'  Rows:    {len(iv_clean):,}')
print(f'  Columns: {list(iv_clean.columns)}')

Saved hcris_iv.csv
  Rows:    2,936
  Columns: ['provider_number', 'price_2011', 'price_2014', 'net_penalty', 'hrrp_penalty', 'vbp_penalty', 'any_penalty', 'net_penalty_1k', 'hrrp_penalty_1k', 'avg_mcare_dis', 'avg_beds', 'avg_mcaid_dis', 'avg_mcare_100', 'price_change']


In [44]:
files = {
    'hcris_clean.csv':             'Full cleaned panel   ',
    'dup_hospitals_by_year.csv':   'Dup hospitals by year  ',
    'unique_hospitals.csv':        'Unique hospital count   ',
    'share_penalized_by_year.csv': 'Share penalized 2012–2019 ',
    'hcris_iv.csv':                'IV cross-section dataset ',
}

print('=== FINAL OUTPUT CHECKLIST ===')
all_good = True
for fname, desc in files.items():
    fpath = OUT_DIR / fname
    if fpath.exists():
        df = pd.read_csv(fpath)
        print(f'  ✅  {fname:<40s}  {len(df):>7,} rows   {desc}')
    else:
        print(f'  ❌  {fname:<40s}  NOT FOUND — rerun cells above')
        all_good = False

print()
if all_good:
    print('All outputs ready.')
else:
    print('N/A')

=== FINAL OUTPUT CHECKLIST ===
  ✅  hcris_clean.csv                            60,518 rows   Full cleaned panel   
  ✅  dup_hospitals_by_year.csv                      11 rows   Dup hospitals by year  
  ✅  unique_hospitals.csv                            1 rows   Unique hospital count   
  ✅  share_penalized_by_year.csv                     8 rows   Share penalized 2012–2019 
  ✅  hcris_iv.csv                                2,936 rows   IV cross-section dataset 

All outputs ready.
